# skforecast Integration Example

This notebook demonstrates how to use tcpfi with skforecast's ForecasterMultiSeries.

**Note:** This notebook requires the `skforecast` optional dependency:
```bash
pip install tcpfi[skforecast]
```

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# Check if skforecast is available
try:
    from skforecast.ForecasterMultiSeries import ForecasterMultiSeries
    SKFORECAST_AVAILABLE = True
except ImportError:
    SKFORECAST_AVAILABLE = False
    print("skforecast not installed. Install with: pip install tcpfi[skforecast]")

## 1. Create Sample Data

In [ ]:
np.random.seed(42)
n_periods = 500
dates = pd.date_range('2023-01-01', periods=n_periods, freq='h')

series_data = {}
for i in range(3):
    series_id = f'MT_{i+1:03d}'
    base = np.random.randn() * 10
    trend = np.linspace(0, 5, n_periods)
    seasonal = 5 * np.sin(2 * np.pi * np.arange(n_periods) / 24)
    noise = np.random.randn(n_periods) * 0.5
    series_data[series_id] = base + trend + seasonal + noise

df = pd.DataFrame(series_data, index=dates)
print(f"Data shape: {df.shape}")
df.head()

## 2. Train ForecasterMultiSeries

In [ ]:
if SKFORECAST_AVAILABLE:
    forecaster = ForecasterMultiSeries(
        regressor=RandomForestRegressor(
            n_estimators=50,
            max_depth=10,
            random_state=42
        ),
        lags=12
    )
    forecaster.fit(series=df)
    print(f"Forecaster fitted with {forecaster.regressor.n_features_in_} features")
else:
    print("Skipping: skforecast not available")

## 3. Create Adapter and Extract Training Data

In [ ]:
if SKFORECAST_AVAILABLE:
    from tcpfi.adapters.skforecast import from_skforecast
    
    adapter = from_skforecast(forecaster)
    X, y = adapter.get_training_data()
    
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Features: {adapter.get_feature_names()}")
    print(f"Series IDs: {adapter.get_series_ids()}")
else:
    print("Skipping: skforecast not available")

## 4. Compute Conditional Importance

In [ ]:
if SKFORECAST_AVAILABLE:
    from tcpfi import ConditionalPermutationImportance
    
    explainer = ConditionalPermutationImportance(
        model=adapter,
        metric='mse',
        strategy='auto',
        n_repeats=3,
        n_jobs=1,
        random_state=42,
    )
    
    result = explainer.compute(X, y)
    print("\nFeature Importance:")
    display(result.to_dataframe())
else:
    print("Skipping: skforecast not available")

## 5. Visualize Results

In [ ]:
if SKFORECAST_AVAILABLE:
    from tcpfi.visualization import plot_importance_bar
    
    fig, ax = plot_importance_bar(
        result,
        max_features=12,
        title='Conditional Permutation Feature Importance (skforecast)'
    )
else:
    print("Skipping: skforecast not available")

## Summary

This notebook demonstrated the seamless integration between tcpfi and skforecast:

1. Train a `ForecasterMultiSeries` model
2. Create a `SkforecastAdapter` using `from_skforecast()`
3. Extract training data via `adapter.get_training_data()`
4. Use the adapter directly with `ConditionalPermutationImportance`

The adapter handles all the complexity of extracting the training matrix and providing predictions in the format expected by tcpfi.